In [8]:
# load dataset
import pandas as pd

df = pd.read_csv("caffeine_crash_and_anxiety_telemetry.csv")

In [9]:
# EDA
print(f"cek 5 data teratas \n {df.head()} \n")

print(f"cek tipe data \n {df.dtypes} \n")

print(f"cek missing values \n {df.isnull().sum()} \n")

print(f"cek duplicate data \n {df.duplicated().sum()}")

cek 5 data teratas 
      user_id  age  gender cyp1a2_metabolizer_type caffeine_tolerance  \
0  CAF-00001   27  Female                    Fast               High   
1  CAF-00002   57  Female                    Fast                Low   
2  CAF-00003   39    Male                  Normal                Low   
3  CAF-00004   34  Female                    Fast           Moderate   
4  CAF-00005   22  Female                    Slow               High   

   baseline_anxiety_score  prior_sleep_hours  daily_hydration_liters  \
0                     2.4                7.0                    2.29   
1                     9.1                7.1                    2.41   
2                     2.9                5.1                    2.07   
3                     3.8                5.0                    2.34   
4                     2.0                6.3                    1.92   

     primary_caffeine_source  caffeine_mg  sugar_grams  \
0         Brewed Drip Coffee          138          8.7 

In [10]:
# cleaning data
df=df.drop(columns='user_id')

In [5]:
# cek outlier
numerical_columns = df.select_dtypes(include='number').columns

outlier_summary = []

for column in numerical_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]

    outlier_summary.append({
        'kolom': column,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'batas_bawah': lower_bound,
        'batas_atas': upper_bound,
        'jumlah_outlier': len(outliers),
        'persen_outlier': round(len(outliers) / len(df) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
outlier_df

,kolom,Q1,Q3,IQR,batas_bawah,batas_atas,jumlah_outlier,persen_outlier
0,age,24.00,40.00,16.00,0.000,64.000,38,0.47
1,baseline_anxiety_score,1.80,4.60,2.80,-2.400,8.800,243,3.04
2,prior_sleep_hours,5.50,7.30,1.80,2.800,10.000,0,0.00
3,daily_hydration_liters,1.56,2.65,1.09,-0.075,4.285,21,0.26
4,caffeine_mg,157.00,234.00,77.00,41.500,349.500,283,3.54
5,sugar_grams,0.80,34.60,33.80,-49.900,85.300,0,0.00
6,consumption_duration_min,10.00,45.00,35.00,-42.500,97.500,2,0.03
7,heart_rate_spike_bpm,2.00,6.20,4.20,-4.300,12.500,424,5.30
8,jitters_intensity_score,1.30,4.50,3.20,-3.500,9.300,228,2.85
9,acute_anxiety_score,1.70,5.10,3.40,-3.400,10.200,0,0.00


In [6]:
# handle outlier
df_capped = df.copy()

for column in numerical_columns:

    Q1 = df_capped[column].quantile(0.25)
    Q3 = df_capped[column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - (1.5 * IQR)
    upper = Q3 + (1.5 * IQR)

    df_capped[column] = df_capped[column].clip(
        lower=lower,
        upper=upper
    )

In [11]:
# encoding
df['gender'] = df['gender'].map({
    'Male': 0,
    'Female': 1
})

df['cyp1a2_metabolizer_type'] = df['cyp1a2_metabolizer_type'].map({
    'Slow': 0,
    'Normal': 1,
    'Fast': 2
})

df['caffeine_tolerance'] = df['caffeine_tolerance'].map({
    'Low': 0,
    'Moderate': 1,
    'High': 2
})

df = pd.get_dummies(
    df,
    columns=['primary_caffeine_source'],
    dtype=int
)

In [ ]:
# split data
x = df.drop(columns = ['crash_risk_level'])
y = df['crash_risk_level']

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42
)